In [7]:
# notebooks/02_baseline.ipynb
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score, classification_report
from sklearn.preprocessing import LabelEncoder
import joblib

# 1. Load data
train = pd.read_csv('../data/train.csv')

# 2. Simple preprocessing
X = train.drop(['id', 'diagnosed_diabetes'], axis=1)
y = train['diagnosed_diabetes']

categorical_cols = ['gender', 'ethnicity', 'education_level', 
                   'income_level', 'smoking_status', 'employment_status']

X_encoded = pd.get_dummies(X, columns=categorical_cols)

# 3. Split train and validation sets
X_train, X_val, y_train, y_val = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42, stratify=y
)

# 4. Train decision tree(Baseline model)
dt_model = DecisionTreeClassifier(
    max_depth=5,
    random_state=42
)
dt_model.fit(X_train, y_train)

# 5. Evaluation
y_pred_proba = dt_model.predict_proba(X_val)[:, 1]
y_pred = dt_model.predict(X_val)

print("="*50)
print("Baseline model - Decision Tree results") 
print("="*50)
print(f"AUC-ROC: {roc_auc_score(y_val, y_pred_proba):.4f}")
print("\nClassification report:") 
print(classification_report(y_val, y_pred))

# 6. Feature importance
feature_importance = pd.DataFrame({
    'feature': X_train.columns,
    'importance': dt_model.feature_importances_
}).sort_values('importance', ascending=False)

print("\nTop 10 important features:") 
print(feature_importance.head(10))

# 7. Save model
joblib.dump(dt_model, '../models/baseline_dt.pkl')


Baseline model - Decision Tree results ==================================================
AUC-ROC: 0.6825

Classification report:              precision    recall  f1-score   support

        0.0       0.60      0.30      0.40     52739
        1.0       0.68      0.88      0.76     87261

avg / total       0.65      0.66      0.63    140000


Top 10 important features:                                feature  importance
15             family_history_diabetes    0.448009
2   physical_activity_minutes_per_week    0.299436
0                                  age    0.217834
6                                  bmi    0.034138
10                          heart_rate    0.000318
14                       triglycerides    0.000264
32           income_level_Lower-Middle    0.000000
26            education_level_Graduate    0.000000
27          education_level_Highschool    0.000000
28           education_level_No formal    0.000000


['../models/baseline_dt.pkl']